In [ ]:
import os
import pandas as pd
import openai
#openai.api_key PUT HERE


import random

#openai.Model.list()

In [ ]:
# Load the radiology report dataframe
df = pd.read_csv("all_11_tasks_round2(6.12).csv")
df.text = df.text.str.replace("\n+","\n")
print(df['bladder abnormal'].value_counts())



In [ ]:
df.MRN = df.MRN.astype(str)

In [ ]:
from tqdm import tqdm

# Define the tasks and labels

task_cols = ['left central calyceal dilation', 'left parenchymal appearance abnormal',
         'left parenchymal thickness abnormal', 'left peripheral calyceal dilation',
         'left ureter abnormal', 'right central calyceal dilation',
         'right parenchymal appearance abnormal', 'right parenchymal thickness abnormal',
         'right peripheral calyceal dilation', 'right ureter abnormal', 'bladder abnormal']

candidate_labels = ['Yes', 'No', 'Cannot answer because of absent kidney or multicystic dysplastic kidney (MDCK)']


In [ ]:
label_dict = {}
label_dict['text'] = df['text'].to_list()

for col in task_cols:
    label_dict[col] = {}
    label_dict[col]['labels'] = df[col].to_list()
    label_dict[col]['preds'] = []

In [ ]:
def generate_prompt(task, examples, example_keys, report, candidate_labels):
    prompt = f"Please read an ultrasound report, and decide if the patient has {task}. "
    prompt += f"You should answer by choosing one of the following labels: {candidate_labels}. I'll provide you with three examples first."
    prompt += f"\n\nExample 1 - Radiology report: \n{examples[0]}\n Correct answer:{example_keys[0]}"
    prompt += f"\n\nExample 2 - Radiology report: \n{examples[1]}\n Correct answer:{example_keys[1]}"
    prompt += f"\n\nExample 3 - Radiology report: \n{examples[2]}\n Correct answer:{example_keys[2]}\n\n"
    prompt += f"Now read the following radiology report:\n\n{report}\n\n"
    prompt += f"Now answer: does the patient have {task}?"
    return prompt 

def generate_prompt_new(task, examples, example_keys, report, candidate_labels):
    task = ", ".join(task)
    prompt = f"Please read an ultrasound report, and decide if the patient has each of the following: {task}. "
    prompt += f"You should answer by choosing one of the following labels for each: {candidate_labels}."
    #prompt += f"\n\nExample 1 - Radiology report: \n{examples[0]}\n Correct answer:{example_keys[0]}"
    #prompt += f"\n\nExample 2 - Radiology report: \n{examples[1]}\n Correct answer:{example_keys[1]}"
    #prompt += f"\n\nExample 3 - Radiology report: \n{examples[2]}\n Correct answer:{example_keys[2]}\n\n"
    prompt += f"Read the following radiology report:\n\n{report}\n\n"
    prompt += f"Now, does the patient have each of the following: {task}?"
    return prompt 


In [ ]:
length = []
label_map = {0: 'Yes', 1: 'No', 2: 'Absent Kidney/MDCK'}
for i, report in tqdm(list(enumerate(df['text']))):#(test['text']))):
    task_results = [] 
    for task in task_cols:
        #print(task)
        example_keys = [1,0,1]#[1,0,2] THERE ARE NO MISSING KIDNEY WITH BLADDER ISSUES
        examples = [df[df[task]==key].sample(1, random_state=42)["text"].tolist()[0] for key in example_keys]
        example_keys = [label_map[i] for i in example_keys]
        prompt = generate_prompt(task, examples, example_keys, report, candidate_labels)
# #         print(prompt)
        length.append(len(prompt.split(" ")))

In [ ]:
print(prompt)



In [ ]:
from matplotlib import pyplot as plt
%matplotlib inline

plt.hist(length, bins=range(min(length), max(length) + 1, 50), 
              alpha=0.4, color="red")
plt.xlabel("length of sentence")
plt.ylabel("proportion")

In [ ]:
# Define a function to run inference on the generated prompts
import time
import json

#uncomment and comment code to use gpt-4o or gpt-5
def run_inference(client, prompt):
    try:

        response = client.chat.completions.create(
            model="gpt-5",
            response_format={ "type": "json_object" },
            messages=[
                {"role": "system", "content": "You are a helpful assistant designed to output JSON. The result should be stored as key 'answer'."},
                {"role": "user", "content": prompt}
            ]
        )
        #response = openai.ChatCompletion.create(
        #    model="gpt-4o",
        #    temperature=0,
        #    top_p=1,
        #    frequency_penalty=0,
        #    presence_penalty=0,
        #    messages=[{
        #        "role": "user",
        #        "content": prompt
        #    }]
        #)
        json_response = json.loads(response.choices[0].message.content)
        #print(json_response)
        return json_response["answer"]
    except Exception as e: 
        print(f"Error {e} occured")
        return ""

client = openai.OpenAI(api_key='PUT_KEY_HERE')

all_results = []
example_keys = [1,0,2]

# Loop through the tasks and generate prompts for each report in the dataframe
for i, report in tqdm(list(enumerate(df['text']))):

    if random.randint(0,17) != 5: continue

    task_results = [i, report] 

    for task in task_cols:
        example_keys = [1,0,1]#[1,0,2]
        examples = [df[df[task]==key].sample(1, random_state=42)["text"].tolist()[0] for key in example_keys]
        example_keys = [label_map[i] for i in example_keys]
        prompt = generate_prompt(task, examples, example_keys, report, candidate_labels)
# #         print(prompt)
        result = run_inference(client, prompt)
        task_results.append(result)
        #time.sleep(10)
    all_results.append(task_results)

In [ ]:
print(all_results)
print(task_results)
print(task_cols)

In [ ]:

'''
# Define a function to run inference on the generated prompts
import time
def run_inference(prompt):
    try:
        response = openai.ChatCompletion.create(
            model="gpt-4o",
            temperature=0,
            top_p=1,
            frequency_penalty=0,
            presence_penalty=0,
            messages=[{
                "role": "user",
                "content": prompt
            }]
        )
        return response["choices"][0]["message"]["content"]
    except: 
        return ""

all_results = []
example_keys = [1,0,2]

# Loop through the tasks and generate prompts for each report in the dataframe
for i, report in tqdm(list(enumerate(test['text']))):
    task = task_cols[-1]
    example_keys = [1,0,2]
    examples = [train[train[task]==key].sample(1, random_state=42)["text"].tolist()[0] for key in example_keys]
    example_keys = [label_map[i] for i in example_keys]
    prompt = generate_prompt(task, examples, example_keys, report, candidate_labels)
# #         print(prompt)
    result = run_inference(prompt)
    task_results.append(result)
    all_results.append(task_results)
    '''

In [ ]:
import numpy as np
results = np.array(all_results)
results.shape

In [ ]:
results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
# df = df.replace({"Yes":1,"No":0})
df.columns = [id] + ["report"] + task_cols
df.to_csv("results/gpt5_test.csv",index=False)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import math

def wilson_confidence_interval(successes, trials, confidence=0.95):
    z = 1.96  # for 95% confidence interval; adjust for other confidence levels
    n = trials
    if n == 0:
        return 0, 0  # No data case
    
    phat = successes / n
    ci_lower = (phat + z*z/(2*n) - z * math.sqrt((phat*(1-phat)+z*z/(4*n))/n)) / (1 + z*z/n)
    ci_upper = (phat + z*z/(2*n) + z * math.sqrt((phat*(1-phat)+z*z/(4*n))/n)) / (1 + z*z/n)
    
    return ci_lower, ci_upper


metric="weighted"

def calc_metric(y_true, y_pred):
    f1, recall, precision, accuracy = [],[],[], []
    for a, b in zip(y_true, y_pred):
            a = [0 if np.isnan(x) else x for x in a]
            b = [0 if np.isnan(x) else x for x in b]
            f1.append(f1_score(a, b, average=metric))
            recall.append(recall_score(a, b, average="macro"))
            precision.append(precision_score(a, b, average="macro"))
            accuracy.append(accuracy_score(a, b))
    return [f1, recall, precision, accuracy]


import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from math import sqrt
from scipy.stats import norm

metric = "weighted"


#def get_wilson_CI(x, alpha=0.05):


# Modified function to calculate metrics and their CIs
def calc_metric_with_ci(y_true, y_pred):
    f1, recall, precision, accuracy = [], [], [], []
    f1_ci, recall_ci, precision_ci, accuracy_ci = [], [], [], []
    
    for a, b in zip(y_true, y_pred):
        a = [0 if np.isnan(x) else x for x in a]
        b = [0 if np.isnan(x) else x for x in b]
        
        # Calculate metrics
        f1_score_val = f1_score(a, b, average=metric)
        recall_val = recall_score(a, b, average="macro")
        precision_val = precision_score(a, b, average="macro")
        accuracy_val = accuracy_score(a, b)
        
        # Append metric values

        #F1 scoring
        # Count successes where y_true == y_pred (for Wilson CI)
        TP = sum([1 for x, y in zip(a, b) if x == y and y == 1])
        TN = sum([1 for x, y in zip(a, b) if x == y and y != 1])
        FP = sum([1 for x, y in zip(a, b) if x != y and y == 1])
        FN = sum([1 for x, y in zip(a, b) if x != y and y != 1])
        total = len(a)
        v = total - TN
        f1_ci_amount = math.sqrt(1.96**2 * (f1_score_val * (1-f1_score_val) * (2-f1_score_val)**2) / (2 * v))
        recall_ci_amount = math.sqrt(1.96**2 * (recall_val * (1-recall_val) * (2-recall_val)**2) / (2 * v))
        precision_ci_amount = math.sqrt(1.96**2 * (precision_val * (1-precision_val) * (2-precision_val)**2) / (2 * v))
        accuracy_ci_amount = math.sqrt(1.96**2 * (accuracy_val * (1-accuracy_val) * (2-accuracy_val)**2) / (2 * v))
        
        f1.append(f'{round(f1_score_val, 4)} [{round(f1_score_val - f1_ci_amount, 4)}, {round(f1_score_val + f1_ci_amount, 4)}]')
        recall.append(f'{round(recall_val, 4)} [{round(recall_val - recall_ci_amount, 4)}, {round(recall_val + recall_ci_amount, 4)}]')
        precision.append(f'{round(precision_val, 4)} [{round(precision_val - precision_ci_amount, 4)}, {round(precision_val + precision_ci_amount, 4)}]')
        accuracy.append(f'{round(accuracy_val, 4)} [{round(accuracy_val - accuracy_ci_amount, 4)}, {round(accuracy_val + accuracy_ci_amount, 4)}]')
        


        f1_ci.append(f1_ci_amount)
        recall_ci.append(recall_ci_amount)
        precision_ci.append(precision_ci_amount)
        accuracy_ci.append(accuracy_ci_amount)
    return [f1, recall, precision, accuracy]

## Example usage:
#y_true = [[1, 0, 1], [1, 1, 0], [0, 0, 1]]
#y_pred = [[1, 0, 0], [1, 1, 0], [0, 1, 1]]

#metrics = calc_metric(y_true, y_pred)
#print("Accuracy scores with CIs:", metrics[6], metrics[7])



In [ ]:
# calc_metric(test[task_cols].values.T, df.values.T)
# np.unique(results)



task_cols = ['left central calyceal dilation', 'left parenchymal appearance abnormal',
         'left parenchymal thickness abnormal', 'left peripheral calyceal dilation',
         'left ureter abnormal', 'right central calyceal dilation',
         'right parenchymal appearance abnormal', 'right parenchymal thickness abnormal',
         'right peripheral calyceal dilation', 'right ureter abnormal','bladder abnormal']


#df = pd.read_csv("data/11_task_0_preds.csv")##
df = pd.read_csv("results/gpt5_bladder.csv")
#if using 11_task_0_preds use this next bit
#df.columns = task_cols

print(df.values.shape)


test = pd.read_csv("all_11_tasks_round2(6.12).csv")
test.text = test.text.str.replace("\n+","\n")
print(test.values.shape)
test = test.iloc[:df.shape[0]]
print(test.values.shape)


print(test[task_cols].isna().sum())
print(df[task_cols].isna().sum())

test = test.fillna(0)
df = df.fillna(0)
print(df.values.shape)
print(test.values.shape)
print(task_cols)


In [ ]:
import numpy as np
# define the values to assign for each condition
values = [1, 0, 2]

# use numpy.select() to assign values based on conditions
for t in task_cols:
    conditions = [df[t].str.contains('Yes'),
                  df[t].str.contains('No'),
                  df[t].str.contains('Not sure|Cannot')]
    df[t] = np.select(conditions, values, default=np.nan)


In [ ]:
#df = pd.read_csv("results/gpt/3shot/task11.csv")


missing = [x for x in task_cols if x not in df.columns]
print(missing)
missing = [x for x in task_cols if x not in test.columns]
print(missing)

In [ ]:
print(df.head())
df = pd.DataFrame(np.array(calc_metric_with_ci(test[task_cols].values.T.tolist(), df[task_cols].values.T.tolist())))
print(df)
df.columns = task_cols
df.index = ["f1", "recall","precision", "accuracy"]
df

In [ ]:
df

In [ ]:
df.to_csv("results/gpt5-3shot_metrics.csv",index=False)

df.to_csv("results/gpt4o-3shot_.csv",index=False)

In [ ]:
li=df[df[task_cols[2]]==1].text.tolist()

In [ ]:
def generate_prompt(task, exmaples, example_keys, report, candidate_labels):
    prompt = f"Please read an ultrasound report, and decide if the patient has {task}. "
    prompt += f"You should answer by choosing one of the following labels: {candidate_labels}. I'll provide you with three examples first."
    prompt += f"\n\nExample 1 - Radiology report: \n{examples[0]}\n Correct answer:{example_keys[0]}"
    prompt += f"\n\nExample 2 - Radiology report: \n{examples[1]}\n Correct answer:{example_keys[1]}"
    prompt += f"\n\nExample 3 - Radiology report: \n{examples[2]}\n Correct answer:{example_keys[2]}\n\n"
    prompt += f"Now read the following radiology report:\n\n{report}\n\n"
    prompt += f"Now answer: does the patient have {task}?"
    return prompt

task = task_cols[2]
label_map = {0: 'Yes', 1: 'No', 2: 'Absent Kidney/MDCK'}
example_keys = [1,0,2]
examples = [train[train[task]==key].sample(1, random_state=42)["text"].tolist()[0] for key in example_keys]
example_keys = [label_map[i] for i in example_keys]
prompt = generate_prompt(task, examples, example_keys, li[5], candidate_labels)